# 01_opentelemetry — OpenTelemetryPlugin: traces + state x-ray, from the box

The shipped `OpenTelemetryPlugin` turns every `machine.run()` into OpenTelemetry signals without touching business code. It emits two independent signals — *Traces* (a span tree) and *Logs* (an x-ray of `state` after each step) — and you choose the backend by passing a provider. In-memory exporters here keep the run deterministic, no collector required.

Companion guide: [OpenTelemetry extension](../../docs/extensions/opentelemetry.md).

In [ ]:
!pip install "aoa-action-machine[otel]"

In [ ]:
import asyncio

from opentelemetry.sdk._logs import LoggerProvider
from opentelemetry.sdk._logs.export import InMemoryLogRecordExporter, SimpleLogRecordProcessor
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import SimpleSpanProcessor
from opentelemetry.sdk.trace.export.in_memory_span_exporter import InMemorySpanExporter
from pydantic import Field

from aoa.action_machine.auth import GuestRole
from aoa.action_machine.context import Context
from aoa.action_machine.domain.base_domain import BaseDomain
from aoa.action_machine.intents.aspects import regular_aspect, summary_aspect
from aoa.action_machine.intents.check_roles import check_roles
from aoa.action_machine.intents.checkers import result_float
from aoa.action_machine.intents.meta import meta
from aoa.action_machine.model import BaseAction, BaseParams, BaseResult
from aoa.action_machine.plugin.open_telemetry import OpenTelemetryPlugin
from aoa.action_machine.runtime.action_product_machine import ActionProductMachine

In [ ]:
class ShopDomain(BaseDomain):
    name = "shop"
    description = "Shop domain"


class PriceParams(BaseParams):
    amount: float = Field(gt=0, description="Base amount")


class PriceResult(BaseResult):
    total: float = Field(description="Total with tax")


@meta(description="Price an order", domain=ShopDomain)
@check_roles(GuestRole)
class PriceOrderAction(BaseAction[PriceParams, PriceResult]):

    @regular_aspect("Apply tax")
    @result_float("with_tax", required=True, min_value=0)
    async def tax_aspect(self, params, state, box, connections):
        return {"with_tax": round(params.amount * 1.2, 2)}

    @summary_aspect("Build result")
    async def build_summary(self, params, state, box, connections):
        return PriceResult(total=state["with_tax"])

## Run

Colab already runs an event loop, so call `await main()` at the top level instead of `asyncio.run(main())`.

In [ ]:
async def main() -> None:
    # User wires the backend; the plugin only emits. In-memory here for determinism.
    spans = InMemorySpanExporter()
    tracer_provider = TracerProvider()
    tracer_provider.add_span_processor(SimpleSpanProcessor(spans))

    logs = InMemoryLogRecordExporter()
    logger_provider = LoggerProvider()
    logger_provider.add_log_record_processor(SimpleLogRecordProcessor(logs))

    plugin = OpenTelemetryPlugin(tracer_provider=tracer_provider, logger_provider=logger_provider)

    machine = ActionProductMachine(plugins=[plugin])
    result = await machine.run(Context(), PriceOrderAction(), PriceParams(amount=100.0))
    print(f"result: total={result.total}\n")

    # Traces — one root span per run + a child span per step:
    print("Spans (timing & structure):")
    for s in spans.get_finished_spans():
        print(f"  {s.name}")

    # Logs — state x-ray: aoa.state.<field> attributes captured after each step:
    print("\nState x-ray (aoa.state.* from log records):")
    for rec in logs.get_finished_logs():
        lr = rec.log_record
        attrs = dict(lr.attributes or {})
        state = {k.removeprefix("aoa.state."): v for k, v in attrs.items() if k.startswith("aoa.state.")}
        if state:
            print(f"  {lr.body}  step={attrs.get('aoa.aspect')}  state={state}")

await main()